# Code

## O - Setup

In [ ]:
pip install wfdb 

In [ ]:
pip install pandas numpy scikit-learn torch matplotlib seaborn umap-learn

In [ ]:
import os, ast, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

import wfdb
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import umap

warnings.filterwarnings("ignore")

## 1 - Hyperparameters

In [ ]:
PTBXL_PATH    = "/kaggle/input/datasets/garethwmch/ptb-xl-1-0-3/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
RESULTS_DIR   = "./results"

# ECG Hyperparameters
SAMPLING_FREQ = 100 # Note: taken from the Kaggle version of the PTB-XL dataset                 
SEQ_LEN = 1000 # 10 seconds, based on standard window, credits to https://litfl.com/ecg-rate-interpretation/
N_LEADS = 12

# Model Hyperparameters
BATCH_SIZE = 64
EPOCHS = 30
LRATE = 0.001
SEED = 17
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
FOCAL_LEADS = {"II": 1, "III": 2, "aVF": 5, "V4": 9, "V5": 10}
LEAD_NAMES  = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
SEX_LABELS  = {0: "Male", 1: "Female"}
SEX_COLORS  = {0: "#179c25", 1: "#8053c9"} 
PRE_R    = 300
POST_R   = 400
BEAT_LEN = PRE_R + POST_R   

# GradCAM Display threshold for GradCAM from the Galanty paper - [TODO] modify this if it doesn't work
GRADCAM_THRESHOLD = 0.2

os.makedirs(RESULTS_DIR, exist_ok = True)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Currently on Device: {DEVICE}") # [GPU Hours] left right now: 15

## 2 - PTB-XL Data Setup

In [ ]:
def is_mi_record(scp_dict, mi_codes):
    return any(code in mi_codes for code in scp_dict)

def is_norm_record(scp_dict, mi_codes, norm_codes):
    has_norm = any(code in norm_codes and liklihood >= 80 for code, liklihood in scp_dict.items())
    has_mi = any(code in mi_codes for code in scp_dict)
    return has_norm and not has_mi

def load_ptbxl_metadata(path):
    # Refer to https://physionet.org/content/ptb-xl/1.0.3/ 
    ptb_database = os.path.join(path, "ptbxl_database.csv")
    df = pd.read_csv(ptb_database , index_col = "ecg_id")
    df.scp_codes = df.scp_codes.apply(ast.literal_eval)
    scp_map = pd.read_csv(os.path.join(path, "scp_statements.csv"), index_col = 0)
    
    mi_codes = set(scp_map[scp_map.diagnostic_class == "MI"].index)
    norm_codes = set(scp_map[scp_map.diagnostic_class == "NORM"].index)
    df["label"] = df.scp_codes.apply(lambda c: 1 if is_mi_record(c, mi_codes) else 0)
    df["is_normal"] = df.scp_codes.apply(lambda c: 1 if is_norm_record(c, norm_codes) else 0)

    df = pd.concat([df[df.label == 1], df[df.is_normal == 1]])
    df = df.dropna(subset = ["sex"])
    df["sex"] = df["sex"].astype(int)

    print(f"PTB-XL Dataset Stats: ")
    # TODO - COME BACK AND CHECK THIS RATIO
    print(f"Total = {len(df)}, Myocardial Infarction (MI) = {df.label.sum()}, Normal ECGs (NORM) = {(df.label==0).sum()}", 
          f"Female = {df.sex.sum()}, Male = {(df.sex==0).sum()}")
    return df

def load_signal(row, path, fs = 100):
    fname = row.filename_lr if fs == 100 else row.filename_hr
    data, headers = wfdb.rdsamp(os.path.join(path, fname))
    return data.T.astype(np.float32) 

def preprocess_signal(signal, seq_len = SEQ_LEN):
    # Ensures that the signal is of the required length - crop if too long, zero pad if too short (Galanty)
    time = signal.shape[1]
    if time >= seq_len:
        signal = signal[:, :seq_len]
    else:
        signal = np.concatenate([signal, np.zeros((12, seq_len - time), dtype = np.float32)], axis = 1)
    mean = signal.mean(axis = 1, keepdims = True)
    stdev = signal.std(axis = 1, keepdims = True) + 1e-8
    z_score = (signal - mean) / (stdev + 1e-8)
    return z_score

## 3 - DataLoader

In [ ]:
class ECGDataset(Dataset):
    # credits to https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html#creating-a-custom-dataset-for-your-files
    def __init__(self, df, path, fs = 100, seq_len = SEQ_LEN):
        self.records = df.reset_index()
        self.path = path
        self.fs = fs
        self.seq_len = seq_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records.iloc[index]
        signal = load_signal(row, self.path, self.fs)
        signal = preprocess_signal(signal, self.seq_len)
        signal_label = torch.tensor(row.label, dtype = torch.float32)

        all_data = (torch.tensor(signal), signal_label, int(row.sex), int(row.ecg_id))
        return all_data

def balance_the_sexes(df):
    # Uses a sampler to make the ratio effectively 50-50, enough said :D
    # Credits to 
    #    https://discuss.pytorch.org/t/using-weightedrandomsampler-for-an-imbalanced-classes/74571/6
    #    https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.WeightedRandomSampler
    # Note: replacement is set to true because women are underrepresented in the PTB dataste
    sex_counts = df.sex.value_counts().to_dict()
    seq_weights = df.sex.map(lambda sex: 1.0/ sex_counts[sex]).values
    sampler = WeightedRandomSampler(
        weights = torch.tensor(seq_weights, dtype = torch.float32), 
        num_samples = len(seq_weights), 
        replacement = True)
    return sampler
  

def make_loaders(df, path, fs = 100, seq_len = SEQ_LEN, batch_size = BATCH_SIZE):
    # Follows the standard 80:10:10 split
    # Credits to https://docs.pytorch.org/docs/stable/data.html#memory-pinning
    train_df = df[df.strat_fold <= 8]
    val_df = df[df.strat_fold == 9]
    test_df = df[df.strat_fold == 10]

    sampler = balance_the_sexes(train_df)

    train_loader = DataLoader(
        ECGDataset(train_df, path, fs, seq_len), 
        batch_size = batch_size, 
        sampler = sampler, 
        pin_memory = True
    )

    val_loader = DataLoader(
        ECGDataset(val_df, path, fs, seq_len), 
        batch_size = batch_size,  
        pin_memory = True, 
        shuffle = False,
    )
    test_loader = DataLoader(
        ECGDataset(test_df, path, fs, seq_len), 
        batch_size = batch_size,  
        pin_memory = True, 
        shuffle = False,
    )
    loaders_dict = {"train": train_loader, "val": val_loader, "test": test_loader}
    num_male = (train_df.sex == 0).sum()
    num_female = train_df.sex.sum()
    print(f"Training Dataset Stats: ")
    print(f"Female = {num_female}, Male = {num_male}")
    return loaders_dict, test_df

## 4 - Model Architectures

### Galanty CNN

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel):
        super().__init__()
        conv1D = nn.Conv1d(in_channels = in_channels, out_channels = out_channels, kernel_size = kernel)
        self.conv_block = nn.Sequential(conv1D, nn.ReLU(), nn.MaxPool1d(2))
        
    def forward(self, x):
        return self.conv_block(x)

class GalantyCNN(nn.Module):
    """
    Follows the architecture of the CNN (Model 1) in "Investigating sex bias in ECG classification for Atrial Fibrillation, Sinus
    Rhythm and Myocardial Infarction"
    """
    def __init__(self, n_leads = N_LEADS, dropout_rate = 0.3):
        
        super().__init__()
        num_hidden = 128 # modified, was 20 in the paper
        num_labels = 1 # modified, was 3 in the paper 
        self.conv_block1 = ConvBlock(n_leads, 64, kernel = 5)
        self.conv_block2 = ConvBlock(64, 64, kernel = 5)
        self.conv_block3 = ConvBlock(64, 128, kernel = 3)
        self.conv_block4 = ConvBlock(128, 128, kernel = 3)

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(num_hidden, num_hidden)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(num_hidden, num_labels)

    def features(self, x):
        # Final output for these 4 blocks would be of the dimensions (B, 128, T')
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)
        return x         

    def get_logits(self, embeddings):
        x = F.relu(self.fc1(embeddings))
        x = self.dropout(x)
        logits = self.fc2(x).squeeze(-1)
        return logits

    def forward(self, x):
        features  = self.features(x)
        embeddings = self.gap(features).squeeze(-1)
        logits = self.get_logits(embeddings)
        return logits

    def get_embeddings(self, x):
        # Final output would be of the dimensions (B, ) and  (B, 128)
        features = self.features(x)
        embeddings = self.gap(features).squeeze(-1)
        logits = self.get_logits(embeddings)
        return logits, embeddings                      


### PatchTST Transformer Model

In [ ]:
class LearnablePositionalEncoding(nn.Module):
    # creds: https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html
    def __init__(self, n_patches, d_model):
        super().__init__()
        self.pe = nn.Embedding(n_patches, d_model)

    def forward(self, x):
        positions = torch.arange(x.size(1), device = x.device)
        concat_pos_info = x + self.pe(positions)
        return concat_pos_info
 
class PatchTSTClassifier(nn.Module):
    # Note: No convolutions so that the 2 models can be compared on basis of architecture difference
    def __init__(self, n_leads = N_LEADS, seq_len = SEQ_LEN, patch_len = PATCH_LEN, 
                 d_model = D_MODEL, n_heads = N_HEADS, n_layers = N_LAYERS, d_ff = D_FF, 
                 dropout = DROPOUT):
        super().__init__()
        self.n_leads = n_leads
        self.n_patches = seq_len//patch_len
        self.patch_len = patch_len
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_enc = LearnablePositionalEncoding(self.n_patches, d_model)
        self.drop = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(d_model = d_model, 
                                                   nhead = n_heads, dim_feedforward = d_ff, 
                                                   dropout = dropout,
                                                   batch_first = True, norm_first = True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers = n_layers)
        embed_dimensions = n_leads * d_model
        self.norm = nn.LayerNorm(embed_dimensions)
        self.head = nn.Linear(embed_dimensions, 1)

    def encode_lead(self, x_lead):
        B = x_lead.size(0)
        x = x_lead.reshape(B, self.n_patches, self.patch_len)
        x = self.patch_embed(x)
        x = self.drop(self.pos_enc(x))
        x = self.transformer(x)
        return x

    def embed(self, x):
        lead_embeddings = []
        for l in range(self.n_leads):
            lead_enc = self.encode_lead(x[:, l, :])
            lead_emb = lead_enc.mean(dim = 1)  
            lead_embeddings.append(lead_emb)
        concatt = torch.cat(lead_embeddings, dim = -1)
        return concatt

    def forward(self, x):
        x = self.embed(x)
        x = self.head(self.norm(x))
        x = x.squeeze(-1)
        return x

    def get_embeddings(self, x):
        embs = self.embed(x)
        res = self.head(self.norm(embs)).squeeze(-1)
        return res,  embs

    def get_attention_for_lead(self, x_single, lead_idx = 1):
        x_lead = x_single[:, lead_idx, :]          
        B = x_lead.size(0)
        tokens = self.drop(self.pos_enc(
            self.patch_embed(x_lead.reshape(B, self.n_patches, self.patch_len))))
        attention_list = []
        hidden = tokens
        for layer in self.transformer.layers:
            h_norm = layer.norm1(hidden)
            attn_out, attn_w = layer.self_attn(h_norm, h_norm, h_norm, need_weight = True,
                                               average_attn_weights = False)
            attention_list.append(attn_w.detach())
            hidden = hidden + layer.dropout1(attn_out)
            hidden = hidden + layer.dropout2(layer.linear2(
                layer.dropout(layer.activation( 
                    layer.linear1(layer.norm2(hidden))))))
        return attention_list  

## 5 - Model Train & Optimization

In [ ]:
def calculate_pos_weight(loader):
    # Credits to https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html
    # "A weight of positive examples" 
    pos = 0 
    neg = 0 
    for signals, labels, sex, ecg_ids in loader:
        pos += labels.sum().item()
        neg += (1 - labels).sum().item()
    ratio = [neg/max(pos, 1)]
    return torch.tensor(ratio)

def train_an_epoch(model, loader, optimizer, criterion): 
    # Credits: 
    #    https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html
    #    https://discuss.pytorch.org/t/about-torch-nn-utils-clip-grad-norm/13873
    model.train()
    total = 0.0
    for signals, labels, sex, ecg_ids in loader:
        signals, labels = signals.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(signals), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    loss_per_batch = total / len(loader)
    return loss_per_batch
        
@torch.no_grad()
def model_evaluate(model, loader):
    model.eval()
    pred_probs, true_labels, sex_tracker, ecg_tracker, embeddings_list = [], [], [], [], []
    for signals, labels, sex, ecg_ids in loader:
        signals = signals.to(DEVICE)
        logits, embeddings = model.get_embeddings(signals)
        prob = torch.sigmoid(logits).cpu().numpy()
        
        pred_probs.append(prob)
        true_labels.append(labels.numpy())

        converted_embeddings = embeddings.cpu().numpy()
        embeddings_list.append(converted_embeddings)
        
        sex_tracker.extend(sex.tolist())
        ecg_tracker.extend(ecg_ids.tolist())

    return (np.concatenate(pred_probs), np.concatenate(true_labels), 
            np.array(sex_tracker), np.array(ecg_tracker), np.vstack(embeddings_list))

def model_train(model, loaders):
    pos_weight = calculate_pos_weight(loaders["train"]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight = pos_weight)

    optimizer = Adam(model.parameters(), lr = LRATE, weight_decay = 0.0001)
    scheduler = CosineAnnealingLR(optimizer, T_max = EPOCHS)

    best_auc, best_model_state = 0.0, None
    record = {"Train Loss": [], "Val AUC": []}
    print("Starting Model Training!")
    for epoch in range(1, EPOCHS + 1):
        loss = train_an_epoch(model, loaders["train"], optimizer, criterion)
        pred_probs, labels, *_ = model_evaluate(model, loaders["val"])
        auc = roc_auc_score(labels, pred_probs)
        scheduler.step()
        record["Train Loss"].append(loss)
        record["Val AUC"].append(auc)
        if auc > best_auc:
            best_auc = auc
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        if epoch % 5 == 0:
            print(f"Epoch #{epoch}, Train Loss {loss:.4f}, Val AUC {auc:.4f}")
    
    model.load_state_dict(best_model_state)
    print(f"Best Val AUC: {best_auc:.4f}")
    return model, record
        
def plot_training_curves(record, save_path = None):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(record["Train Loss"], color = SEX_COLORS[0], lw = 1.8)
    axes[0].set_title("Training Loss"); axes[0].set_xlabel("Epoch #")
    axes[0].set_ylabel("BCE Loss")
    axes[0].spines[["top","right"]].set_visible(False)

    axes[1].plot(record["Val AUC"], color = SEX_COLORS[1], lw = 1.8)
    axes[1].axhline(max(record["Val AUC"]), color = "gray",
                    ls ="--", lw = 0.8, alpha = 0.5,
                    label = f"Best = {max(record['Val AUC']):.3f}")
    axes[1].set_title("Validation AUC"); axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("AUC"); axes[1].legend(fontsize=9)
    axes[1].spines[["top","right"]].set_visible(False)
    fig.suptitle("Model 1: Galanty CNN", fontsize = 12)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi = 200, bbox_inches = "tight")
        print(f"Saved!! {save_path}")
    plt.show()
    plt.close(fig)

In [ ]:
df = load_ptbxl_metadata(PTBXL_PATH)
loaders, test_df = make_loaders(df, PTBXL_PATH)

model = GalantyCNN(n_leads = N_LEADS).to(DEVICE) # note - substitute here with the model from approach 1 or 2
model, history = model_train(model, loaders)
torch.save(model.state_dict(), f"{RESULTS_DIR}/model.pt")
plot_training_curves(history, save_path =f"{RESULTS_DIR}/training_curves_figure.png")
probs, labels, sex, ecg_ids, embeddings = model_evaluate(model, loaders["test"])
print(f"Overall Test AUC: {roc_auc_score(labels, probs):.4f}")